In [ ]:
# Télecharger les séries temporelles par bâtiment depuis le data lake public OEDI (ResStock 2025, AMY2018)
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_RAW       = ROOT / 'data' / 'raw'


df_metadata = pd.read_parquet(DATA_PROCESSED / 'metadata_features.parquet')
OEDI_BASE = (
    'https://oedi-data-lake.s3.amazonaws.com/'
    'nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/'
    '2025/resstock_amy2018_release_1/'
    'timeseries_individual_buildings/by_state/upgrade=0'
)

def download_timeseries(bldg_id: int, state: str, save: bool = True) -> pd.DataFrame:
    out = DATA_RAW / f'{bldg_id}-0.parquet'
    if out.exists():    #si le fichier existe déjà on charge localement sans retélécharger
        print(f'Fichier existant trouvé, chargement local : {out}')
        df = pd.read_parquet(out)
    else:
        url = f'{OEDI_BASE}/state={state}/{bldg_id}-0.parquet'
        print(f'Téléchargement : {url}')
        df = pd.read_parquet(url)
        if save:
            #out.parent.mkdir(parents=True, exist_ok=True) 
            df.to_parquet(out)
            print(f'Sauvegardé : {out}')
    print(f'Shape : {df.shape} | {df["timestamp"].iloc[0]} → {df["timestamp"].iloc[-1]}')
    return df

In [ ]:
CONSO_COL = 'out.electricity.total.energy_consumption..kwh'
CHAUF_COL = 'out.electricity.heating.energy_consumption..kwh'
CLIM_COL  = 'out.electricity.cooling.energy_consumption..kwh'
PIC_ETE   = 'out.qoi.electricity.maximum_daily_peak_summer..kw'
PIC_HIVER = 'out.qoi.electricity.maximum_daily_peak_winter..kw'

for col in [CONSO_COL, CHAUF_COL, CLIM_COL, PIC_ETE, PIC_HIVER]:
    if col in df_metadata.columns:
        df_metadata[col] = pd.to_numeric(df_metadata[col], errors='coerce')

par_etat = df_metadata.groupby('in.state', as_index=False).agg(nb=(CONSO_COL, 'count'),conso_moy=(CONSO_COL, 'mean')).sort_values('nb', ascending=False).head(20)
par_etat['coef'] = round(par_etat['nb'] / par_etat['nb'].mean())
par_etat = par_etat.reset_index(drop=True)
par_etat

In [ ]:
from random import randint

choisis = []
choisis_ind = []

for i in range(par_etat.shape[0]):
    temp = []
    while len(temp)!= par_etat['coef'].iloc[i]:
        indice_alea = randint(0,df_metadata.shape[0])
        bldg_line = df_metadata.loc[indice_alea]

        electricity = df_metadata['out.electricity.total.energy_consumption..kwh'].loc[indice_alea]
        fuel_oil = df_metadata['out.fuel_oil.total.energy_consumption..kwh'].loc[indice_alea]
        natural_gas = df_metadata['out.natural_gas.total.energy_consumption..kwh'].loc[indice_alea]
        propane = df_metadata['out.propane.total.energy_consumption..kwh'].loc[indice_alea]
        non_electric_sources = fuel_oil + natural_gas +propane
    
        NO_POOL = bldg_line['in.misc_pool']!='Has Pool'
        NO_EV = bool(bldg_line['in.electric_vehicle_ownership']==0)
        NO_PV = bool(bldg_line['in.has_pv']==0)
        HAS_ELEC_HEAT = bool(bldg_line['out.electricity.heating.energy_consumption..kwh']!=0)
        NO_OTHER = bool(non_electric_sources==0)
        ONLY_ELEC = bool(electricity!= 0)
        IN_STATE = bool(bldg_line['in.state']==par_etat['in.state'].iloc[i])

        if NO_POOL and NO_EV and NO_PV and HAS_ELEC_HEAT and NO_OTHER and ONLY_ELEC and IN_STATE :
            temp.append(int(df_metadata['bldg_id'].loc[indice_alea]))
            choisis_ind.append(indice_alea)
    for e in temp:
        choisis.append(e)

print(choisis)
print(choisis_ind)

In [ ]:
#Variables obtenues en executant le code ci-dessus la première fois

choisis = [13577, 246060, 530175, 321644, 486019, 124910, 10163, 245026, 521854, 290410, 115045, 397032, 308989, 86159, 541960, 535861, 347201, 207764, 463378, 182255, 347219, 542226, 501484, 475623,336]
choisis_ind = [13576, 246046, 530145, 321627, 485989, 124904, 10162, 245012, 521824, 290393, 115039, 397011, 308972, 86153, 541930, 535831, 347183, 207750, 463350, 182243, 347201, 542196, 501454, 475595,337]
len(choisis)==par_etat['coef'].sum()

In [ ]:
states = []
for i in choisis_ind:
    states.append(df_metadata['in.state'].loc[i])

# Crée un dataframe pour pouvoir retrouver les bâtiments choisis sur la page ReStock
d = {"index_in_metadata": choisis_ind, "bldg_id": choisis, "state": states}
df_choisis = pd.DataFrame(data=d)
df_choisis    